# 05.6 - Random Forests

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

A single decision tree overfits and is unstable. A **random forest** trains many trees on bootstrap samples with random feature subsets, then averages their predictions. This reduces variance dramatically.

## 2. Why Does This Matter?

Random forests are among the best off-the-shelf models for tabular data. They handle nonlinearity, interactions, and mixed features with little tuning.

## 3. Prerequisites

- Unit 05.5 (Decision Trees)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain bagging and random feature selection
- Build and evaluate a random forest
- Compare a single tree to a forest
- Understand feature importance

## 5. Mental Model

Bagging (Bootstrap Aggregating):

1. Draw B bootstrap samples (with replacement) from training data.
2. Train a tree on each sample, using a random subset of features at each split.
3. Average predictions (regression) or majority vote (classification).

Randomness decorrelates trees, so averaging reduces variance without increasing bias much.


## 6. Generate Data

Use a classification dataset.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


Train: 700, Test: 300


## 7. Single Tree vs Random Forest

Compare a single deep tree to a random forest.


In [2]:
# Single deep tree
tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
tree_train = accuracy_score(y_train, tree.predict(X_train))
tree_test = accuracy_score(y_test, tree.predict(X_test))

# Random forest
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_train = accuracy_score(y_train, forest.predict(X_train))
forest_test = accuracy_score(y_test, forest.predict(X_test))

print(f"Single tree:  train={tree_train:.3f}, test={tree_test:.3f}")
print(f"Random forest: train={forest_train:.3f}, test={forest_test:.3f}")
print("\nThe forest generalizes better (higher test accuracy).")


Single tree:  train=1.000, test=0.817
Random forest: train=1.000, test=0.917

The forest generalizes better (higher test accuracy).


## 8. Effect of Number of Trees

More trees reduce variance, but with diminishing returns.


In [3]:
for n in [1, 5, 10, 50, 100, 200]:
    f = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1).fit(X_train, y_train)
    acc = accuracy_score(y_test, f.predict(X_test))
    print(f"n_estimators={n:3d}: test accuracy={acc:.3f}")


n_estimators=  1: test accuracy=0.780
n_estimators=  5: test accuracy=0.870


n_estimators= 10: test accuracy=0.907


n_estimators= 50: test accuracy=0.910


n_estimators=100: test accuracy=0.917


n_estimators=200: test accuracy=0.920


## 9. Feature Importance

Random forests provide feature importance scores.


In [4]:
importances = forest.feature_importances_
top = np.argsort(importances)[::-1][:10]
print("Top 10 features by importance:")
for i in top:
    print(f"  Feature {i}: {importances[i]:.3f}")
print("\nImportance sums to 1 and reflects how much each feature reduces impurity.")


Top 10 features by importance:
  Feature 11: 0.116
  Feature 14: 0.115
  Feature 17: 0.086
  Feature 15: 0.085
  Feature 7: 0.069
  Feature 2: 0.060
  Feature 18: 0.052
  Feature 1: 0.049
  Feature 9: 0.047
  Feature 3: 0.045

Importance sums to 1 and reflects how much each feature reduces impurity.


## 10. Out-of-Bag (OOB) Score

Bootstrap samples leave out ~37% of data; the OOB score estimates performance without a separate validation set.


In [5]:
forest_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
forest_oob.fit(X_train, y_train)
print(f"OOB score: {forest_oob.oob_score_:.3f}")
print(f"Test accuracy: {accuracy_score(y_test, forest_oob.predict(X_test)):.3f}")
print("\nOOB score is a good estimate of test performance.")


OOB score: 0.917
Test accuracy: 0.917

OOB score is a good estimate of test performance.


## 11. Failure Case: Too Few Trees

With few trees, the forest behaves like a single unstable tree.


In [6]:
f1 = RandomForestClassifier(n_estimators=1, random_state=42).fit(X_train, y_train)
print(f"1 tree:  test={accuracy_score(y_test, f1.predict(X_test)):.3f}")
print(f"100 trees: test={forest_test:.3f}")
print("\nMore trees = more stable, better generalization.")


1 tree:  test=0.780
100 trees: test=0.917

More trees = more stable, better generalization.


## 12. Debugging: Common Errors

- **Too few trees**: high variance.
- **Too many features per split**: trees too correlated.
- **Imbalanced data**: use class_weight.

## 13. Real-World Considerations

- Random forests handle missing values and mixed data well.
- They are harder to interpret than a single tree.
- Use n_jobs=-1 for speed.

## 14. Common Mistakes

- Using too few trees.
- Expecting a single tree's interpretability.

## 15. When NOT to Use

- When you need a simple interpretable model.
- When data is very high-dimensional and sparse.

## 16. Challenge

Build a random forest for regression and compare to a single regression tree.


In [7]:
# Challenge: random forest regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

np.random.seed(5)
Xr = np.random.uniform(0, 10, (500, 3))
yr = 2 * Xr[:, 0] - 1.5 * Xr[:, 1] + 0.5 * Xr[:, 2] + np.random.normal(0, 1, 500)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.3, random_state=42)

tr = DecisionTreeRegressor(random_state=42).fit(Xr_tr, yr_tr)
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(Xr_tr, yr_tr)

print(f"Single tree MSE:  {mean_squared_error(yr_te, tr.predict(Xr_te)):.3f}")
print(f"Random forest MSE: {mean_squared_error(yr_te, rf.predict(Xr_te)):.3f}")
print("\nThe forest reduces error by averaging many trees.")


Single tree MSE:  4.458
Random forest MSE: 2.062

The forest reduces error by averaging many trees.


## 17. Closed-Book Recall

Without looking back:

1. What is bagging?
2. Why does random feature selection help?
3. What is the OOB score?
4. Why do forests generalize better than single trees?

## 18. Teach-Back Questions

Explain to another person:

- How a random forest reduces variance.
- The role of randomness in a forest.

## 19. Summary

You built and evaluated random forests, compared them to single trees, and explored feature importance and OOB scores. Forests are powerful, robust models.

## 20. Further Experiment

- Tune max_features.
- Compare to gradient boosting.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
